In [6]:
import cv2
import mediapipe as mp
import math
import numpy as np
import time    
import requests  


WRITE_API_KEY = "5KG2CNCWI5PVZYGT" 
BASE_URL = "https://api.thingspeak.com/update"
UPLOAD_INTERVAL = 16

# --- AI CONFIGURATION ---
BUFFER_SIZE = 30        # Number of frames to analyze
RISK_THRESHOLD = 60     # Score needed to trigger CNN
CNN_CONFIDENCE = 0.8    # Confidence needed from CNN to sound alarm

# Initialize MediaPipe
mpPose = mp.solutions.pose
pose = mpPose.Pose()
mpDraw = mp.solutions.drawing_utils

# Initialize Timer Variables
last_upload_time = 0

# Mock CNN Model (Replace with actual model loading)
class MockCNN:
    def predict(self, data):
        # Simulates CNN analysis. 
        # In reality, pass 'data' to your model.predict()
        return [[0.1, 0.9]] 
cnn_model = MockCNN()

# --- HELPER FUNCTIONS ---
def calculate_distance(p1, p2):
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def calculate_angle(a, b):
    dx = b[0] - a[0]
    dy = b[1] - a[1]
    theta = math.degrees(math.atan2(dy, dx))
    return abs(theta)

def send_alert_to_cloud(is_drowning):
    try:
        # field3 is your Alert Status (1 = Danger, 0 = Safe)
        status_value = 1 if is_drowning else 0
        
        # Construct the URL parameters
        payload = {
            'api_key': WRITE_API_KEY,
            'field3': status_value
        }
        
        # Send the data
        print(f"📡 Sending Status: {status_value} to Cloud...")
        requests.get(BASE_URL, params=payload)
        print("✅ Data Sent Successfully!")
        
    except Exception as e:
        print(f"⚠️ Internet Error: {e}")

def analyze_buffer(buffer):
    risk_score = 0
    
    # 1. Average Limb Speed
    total_diff = 0
    for i in range(1, len(buffer)):
        curr = buffer[i][16] # Right Wrist
        prev = buffer[i-1][16]
        total_diff += calculate_distance(prev, curr)
    
    avg_speed = total_diff / (len(buffer) - 1)
    if avg_speed > 15: risk_score += 30
    
    # 2. Body Angle
    last_frame = buffer[-1]
    shoulder = last_frame[12]
    hip = last_frame[24]
    angle = calculate_angle(shoulder, hip)
    
    if angle > 70: risk_score += 30 # Vertical
    elif angle < 30: risk_score -= 20 # Horizontal
    
    # 3. Head Bobbing
    bob_count = 0
    shoulder_y_avg = sum([f[12][1] for f in buffer]) / len(buffer)
    for i in range(1, len(buffer)):
        nose_curr = buffer[i][0][1]
        nose_prev = buffer[i-1][0][1]
        if nose_curr > shoulder_y_avg and nose_prev <= shoulder_y_avg:
            bob_count += 1
    if bob_count >= 2: risk_score += 20

    # 4. Net Displacement
    start_hip = buffer[0][24]
    end_hip = buffer[-1][24]
    net_disp = calculate_distance(start_hip, end_hip)
    if avg_speed > 15 and net_disp < 20: risk_score += 20

    return risk_score, avg_speed

# --- MAIN LOOP ---
camera = cv2.VideoCapture(0)
landmark_buffer = [] 

print("🚀 System Active. Monitoring...")

while True:
    success, img = camera.read()
    if not success: break
    
    h, w, c = img.shape
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(imgRGB)

    final_status = "Scanning..."
    status_color = (255, 255, 0) # Cyan
    
    # Default detection state for this frame
    d_detected = False 

    if results.pose_landmarks:
        mpDraw.draw_landmarks(img, results.pose_landmarks, mpPose.POSE_CONNECTIONS)
        
        # Extract coordinates
        current_frame_coords = []
        for lm in results.pose_landmarks.landmark:
            current_frame_coords.append((int(lm.x * w), int(lm.y * h)))
            
        landmark_buffer.append(current_frame_coords)
        
        # Maintain Buffer Size
        if len(landmark_buffer) > BUFFER_SIZE:
            landmark_buffer.pop(0)
            
        # Analyze if Buffer Full
        if len(landmark_buffer) == BUFFER_SIZE:
            risk, speed = analyze_buffer(landmark_buffer)
            
            # Logic Chain
            if risk > RISK_THRESHOLD:
                final_status = "Risk High: Verifying..."
                status_color = (0, 165, 255) # Orange
                
                # Mock CNN Prediction
                cnn_result = cnn_model.predict(landmark_buffer)
                drowning_prob = cnn_result[0][1]
                
                if drowning_prob > CNN_CONFIDENCE:
                    final_status = "CRITICAL: DROWNING!"
                    status_color = (0, 0, 255) # Red
                    d_detected = True # <--- THIS IS THE TRIGGER
            else:
                final_status = f"Safe (Risk: {risk})"
                status_color = (0, 255, 0) # Green

            # Debug Info
            cv2.putText(img, f"Speed: {int(speed)}", (10, 150), cv2.FONT_HERSHEY_PLAIN, 1, (255,255,255), 1)

    # UI Display
    cv2.putText(img, final_status, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)
    
    # --- CLOUD SYNC LOGIC (The Bridge) ---
    current_time = time.time()
    
    # Check if 16 seconds have passed since last upload
    if (current_time - last_upload_time) > UPLOAD_INTERVAL:
        
        if d_detected:
            print("❗ Critical: Drowning Detected! Updating App...")
            send_alert_to_cloud(is_drowning=True)
            last_upload_time = current_time 
            
        else:
            # Optional: Send "Safe" status periodically
            # print("🛡️ Area Safe. Updating Cloud...") 
            send_alert_to_cloud(is_drowning=False)
            last_upload_time = current_time 
    
    cv2.imshow("Hybrid Drowning Detection", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camera.release()
cv2.destroyAllWindows()

🚀 System Active. Monitoring...
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
❗ Critical: Drowning Detected! Updating App...
📡 Sending Status: 1 to Cloud...
✅ Data Sent Successfully!


In [ ]:
import cv2
import mediapipe as mp
import math
import numpy as np
import time
import requests

# --- CONFIGURATION: CLOUD ---
WRITE_API_KEY = "5KG2CNCWI5PVZYGT"
BASE_URL = "https://api.thingspeak.com/update"
UPLOAD_INTERVAL = 16  # ThingSpeak needs 15s gap


BUFFER_SIZE = 30        
RISK_THRESHOLD = 50     
VOTE_WINDOW = 30       
VOTE_THRESHOLD = 20     

# --- INITIALIZATION ---
mpPose = mp.solutions.pose
pose = mpPose.Pose()
mpDraw = mp.solutions.drawing_utils

last_upload_time = 0
landmark_buffer = []    # Stores coordinates for AI analysis
detection_history = []  # Stores True/False votes for stability

# Mock CNN (Replace with real model later)
class MockCNN:
    def predict(self, data): return [[0.1, 0.9]] 
cnn_model = MockCNN()

# --- HELPER FUNCTIONS ---
def calculate_distance(p1, p2):
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def calculate_angle(a, b):
    dx = b[0] - a[0]
    dy = b[1] - a[1]
    return abs(math.degrees(math.atan2(dy, dx)))

def send_alert_to_cloud(is_drowning):
    """Sends signal to Flutter App via ThingSpeak"""
    try:
        status_value = 1 if is_drowning else 0
        payload = {'api_key': WRITE_API_KEY, 'field3': status_value}
        
        print(f"📡 Sending Status: {status_value} to Cloud...")
        requests.get(BASE_URL, params=payload)
        print("✅ Data Sent Successfully!")
    except Exception as e:
        print(f"⚠️ Internet Error: {e}")

def analyze_buffer(buffer):
    """Your Custom Logic: Speed, Angle, Bobbing"""
    risk_score = 0
    
    # 1. Average Speed
    total_diff = 0
    for i in range(1, len(buffer)):
        total_diff += calculate_distance(buffer[i-1][16], buffer[i][16])
    avg_speed = total_diff / (len(buffer) - 1)
    if avg_speed > 15: risk_score += 30
    
    # 2. Body Angle
    angle = calculate_angle(buffer[-1][12], buffer[-1][24])
    if angle > 70: risk_score += 30
    elif angle < 30: risk_score -= 20
    
    # 3. Head Bobbing
    bob_count = 0
    shoulder_y_avg = sum([f[12][1] for f in buffer]) / len(buffer)
    for i in range(1, len(buffer)):
        if buffer[i][0][1] > shoulder_y_avg and buffer[i-1][0][1] <= shoulder_y_avg:
            bob_count += 1
    if bob_count >= 2: risk_score += 20

    # 4. Net Displacement (Struggling in place)
    net_disp = calculate_distance(buffer[0][24], buffer[-1][24])
    if avg_speed > 15 and net_disp < 20: risk_score += 20

    return risk_score, avg_speed

# --- MAIN LOOP ---
camera = cv2.VideoCapture(0)
print("🚀 AI Guard Active. Monitoring...")

try:
    while True:
        success, img = camera.read()
        if not success: break
        
        h, w, c = img.shape
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = pose.process(imgRGB)

        # Frame Status Variables
        frame_status_text = "Scanning..."
        frame_color = (255, 255, 0)
        drowning_detected_this_frame = False # Raw signal

        if results.pose_landmarks:
            mpDraw.draw_landmarks(img, results.pose_landmarks, mpPose.POSE_CONNECTIONS)
            
            # 1. Extract Landmarks
            current_frame_coords = []
            for lm in results.pose_landmarks.landmark:
                current_frame_coords.append((int(lm.x * w), int(lm.y * h)))
            landmark_buffer.append(current_frame_coords)
            
            # Maintain Buffer
            if len(landmark_buffer) > BUFFER_SIZE: landmark_buffer.pop(0)
            
            # 2. Analyze (If Buffer Full)
            if len(landmark_buffer) == BUFFER_SIZE:
                risk, speed = analyze_buffer(landmark_buffer)
                
                # Check Risk Threshold
                if risk > RISK_THRESHOLD:
                    # Optional: Verify with CNN
                    cnn_result = cnn_model.predict(landmark_buffer)
                    if cnn_result[0][1] > 0.8: # Confidence check
                        frame_status_text = "RISK HIGH!"
                        frame_color = (0, 0, 255)
                        drowning_detected_this_frame = True # Mark this frame as danger

        # --- 3. VOTING SYSTEM (Temporal Smoothing) ---
        # Add current frame result to history
        detection_history.append(drowning_detected_this_frame)
        if len(detection_history) > VOTE_WINDOW: detection_history.pop(0)

        # Count votes
        drowning_votes = detection_history.count(True)
        
        # Decide Confirmed Status
        confirmed_drowning = False
        if drowning_votes > VOTE_THRESHOLD:
            confirmed_drowning = True
            # FIXED FONT ERROR HERE:
            cv2.putText(img, f"CONFIRMED DROWNING ({drowning_votes}/{VOTE_WINDOW})", (10, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

        # --- 4. CLOUD BRIDGE ---
        current_time = time.time()
        if (current_time - last_upload_time) > UPLOAD_INTERVAL:
            
            if confirmed_drowning:
                print(f"❗ CRITICAL ALERT: Drowning Confirmed. Votes: {drowning_votes}")
                send_alert_to_cloud(is_drowning=True)
                last_upload_time = current_time
            else:
                # Only send safe if very calm (low votes)
                if drowning_votes < 5:
                   # print("🛡️ Status: Safe") 
                   send_alert_to_cloud(is_drowning=False)
                   last_upload_time = current_time

        # Debug UI
        cv2.putText(img, frame_status_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, frame_color, 2)
        cv2.imshow("Sabarmati AI Guard", img)
        
        if cv2.waitKey(1) & 0xFF == ord('q'): break

except KeyboardInterrupt:
    print("Stopping...")
finally:
    camera.release()
    cv2.destroyAllWindows()

🚀 AI Guard Active. Monitoring...
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
❗ CRITICAL ALERT: Drowning Confirmed. Votes: 30
📡 Sending Status: 1 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Successfully!
📡 Sending Status: 0 to Cloud...
✅ Data Sent Succe